In [ ]:
from essential.gpu_utils import select_best_gpus

select_best_gpus()

import scanpy as sc
from sklearn.cluster import KMeans
import plotnine as gg
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE as TSNE_sklearn
from sklearn.cluster import KMeans
from tqdm import tqdm
import itertools
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
import seaborn as sns
from sklearn.metrics import precision_recall_curve

# from sklearn.manifold import TSNE
from cuml.manifold import TSNE
import numpy as np
import pandas as pd
import seaborn as sns
import jax.numpy as jnp
import plotly.express as px
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import scipy.stats as stats

from essential.stats import MMDTestJax, MMDTest
from essential.data import load_fitness_data
from essential.fba import load_ecoli_rich_medium_model
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
import matplotlib.colors as mcolors
import json

pd.set_option("display.max_columns", 500)

SHARED_THEME = gg.theme(
    axis_text=gg.element_text(size=6),
    axis_title=gg.element_text(size=7),
    figure_size=(3, 2),
    title=gg.element_text(size=7),
    legend_text=gg.element_text(size=6),
)


def compute_pairwise(df1, df2=None, metric="cosine"):
    if metric == "cosine":
        metric_func = cosine_similarity
    elif metric == "euclidean":
        metric_func = euclidean_distances
    else:
        raise ValueError(f"Metric {metric} not supported")

    if df2 is None:
        pairwise_d = metric_func(df1)
        pairwise_d = pd.DataFrame(pairwise_d, index=df1.index, columns=df1.index)
    else:
        pairwise_d = metric_func(df1, df2)
        pairwise_d = pd.DataFrame(pairwise_d, index=df1.index, columns=df2.index)
    return pairwise_d


def plot_similarity_matrix(
    df, row_metadata, row_color_column, cmap="viridis", similarity_label="cosine similarity"
):
    """
    Plots a seaborn clustermap with a row colorbar.

    df: pd.DataFrame representing the similarity matrix.
    row_metadata: pd.DataFrame containing metadata for rows (must match df's rows).
    row_color_column: str, the column in row_metadata to map to colors.
    """
    import matplotlib.pyplot as plt
    import matplotlib.colors as mcolors
    import matplotlib.cm as cm
    import seaborn as sns

    cmap_obj = plt.get_cmap(cmap)
    norm = mcolors.Normalize(
        vmin=row_metadata[row_color_column].min(), vmax=row_metadata[row_color_column].max()
    )
    row_colors = row_metadata[row_color_column].map(lambda x: mcolors.to_hex(cmap_obj(norm(x))))

    g = sns.clustermap(
        df,
        row_colors=row_colors,
        xticklabels=False,
        yticklabels=False,
        cbar_pos=(0.02, 0.8, 0.05, 0.18),
        cbar_kws={"label": similarity_label},
    )

    cbar_ax = g.fig.add_axes([0.02, 0.55, 0.05, 0.18])
    cb = plt.colorbar(cm.ScalarMappable(norm=norm, cmap=cmap_obj), cax=cbar_ax)
    cb.set_label(row_color_column)

    return g


def hamming_distance(B: np.ndarray) -> np.ndarray:
    B_ = np.array(B)
    s = B_.sum(axis=1, keepdims=True)
    return s + s.T - 2 * (B_ @ B_.T)


def to_long_no_diagonal(df):
    return (
        df.stack()
        .reset_index()
        .rename(columns={"level_0": "gene1", "level_1": "gene2", 0: "distance"})
        .loc[lambda x: x["gene1"] != x["gene2"]]
        .assign(
            gene_pair=lambda x: x["gene1"] + "_" + x["gene2"],
        )
        .drop_duplicates(subset=["gene_pair"], keep="first")
    )

### Data imports

In [ ]:
fitness_df = load_fitness_data()
fitness_df_gene = fitness_df.groupby("gene")[["T1", "T2", "T3", "T4"]].mean()

flux_df = pd.read_csv("/workspace/experiments/01232026_fba/data/moma_fluxes.csv", index_col=0)
worker_df = pd.read_csv("/workspace/experiments/01232026_fba/data/worker_ids.csv", index_col=0)
wt_flux = pd.read_csv("/workspace/experiments/01232026_fba/data/wt_fluxes_0.csv", index_col=0)
growth_df = (
    pd.read_csv("/workspace/experiments/01232026_fba/data/fba_growth_ratios.csv", index_col=0)
    .merge(fitness_df_gene, left_index=True, right_index=True)
    .assign(
        fba_growth_type=lambda x: pd.Categorical(
            np.where(x["growth_ratio"] >= 0.5, "high", "low"), categories=["low", "high"]
        ),
        growth_score=lambda x: (x["growth"] - x["growth_wt"]) / x["growth_wt"],
        is_predicted_essential=lambda x: x["growth_ratio"] < 0.5,
    )
    .merge(worker_df, left_index=True, right_index=True)
)

flux_df_bin = (flux_df.abs() >= 1e-6).astype(float)
flux_ham_dist = hamming_distance(flux_df_bin)
flux_ham_dist_df = pd.DataFrame(flux_ham_dist, index=flux_df_bin.index, columns=flux_df_bin.index)

In [ ]:
adata = sc.read_h5ad(
    "/workspace/data/251117_genomescale_CRISPRi/sample_mix_umi200_hvg500_pc25_neighbors10_mindist0.55.scvi.h5ad"
)
adata_case = sc.read_h5ad("/workspace/data/251117_genomescale_CRISPRi/adata_case.annotated.h5ad")

transcript_df = []
transcript_case_df = []
z_transcript_df = []
z_transcript_case_df = []
gene_names = []
gene_case_names = []


for gene in tqdm(adata.obs["gene"].unique()):
    adata_gene = adata[adata.obs["gene"] == gene]
    X_gene = adata_gene.layers["cp10k"].toarray()
    if X_gene.shape[0] > 0:
        gene_names.append(gene)
        transcript_df.append(X_gene.mean(axis=0))
        z_transcript_df.append(adata_gene.obsm["X_scVI"].mean(axis=0))

    adata_gene_case = adata_case[adata_case.obs["gene"] == gene]
    X_gene_case = adata_gene_case.layers["cp10k"].toarray()
    if X_gene_case.shape[0] > 0:
        gene_case_names.append(gene)
        transcript_case_df.append(X_gene_case.mean(axis=0))
        z_transcript_case_df.append(adata_gene_case.obsm["X_scVI"].mean(axis=0))
transcript_df = pd.DataFrame(transcript_df, index=gene_names)
transcript_case_df = pd.DataFrame(transcript_case_df, index=gene_case_names)
z_transcript_df = pd.DataFrame(z_transcript_df, index=gene_names)
z_transcript_case_df = pd.DataFrame(z_transcript_case_df, index=gene_case_names)

In [ ]:
# transcript_df_pca = PCA(n_components=50).fit_transform(transcript_df)
# transcript_df_pca_ = pd.DataFrame(transcript_df_pca, index=gene_names)
# transcript_pairwise = compute_pairwise(transcript_df_pca_, metric="euclidean")

# transcript_df_pca = PCA(n_components=50).fit_transform(transcript_case_df)
# transcript_df_pca_ = pd.DataFrame(transcript_df_pca, index=gene_case_names)
# transcript_pairwise = compute_pairwise(transcript_df_pca_, metric="euclidean")

transcript_pairwise = compute_pairwise(z_transcript_case_df, metric="euclidean")

### Exploration

In [ ]:
flux_ham_dist_df_long = (
    flux_ham_dist_df.stack()
    .reset_index()
    .rename(columns={"level_0": "gene1", "level_1": "gene2", 0: "hamming_distance"})
)

min_hamming_dist = (
    flux_ham_dist_df_long.loc[lambda x: x["gene1"] != x["gene2"]]
    .groupby("gene1")["hamming_distance"]
    .min()
    .to_frame("min_hamming")
    .merge(growth_df, left_index=True, right_index=True)
)

fig = (
    gg.ggplot(min_hamming_dist, gg.aes(x="min_hamming", color="is_predicted_essential"))
    + gg.stat_ecdf()
    + gg.theme_minimal()
    + gg.labs(x="Minimum Hamming distance", y="Cumulative Density")
    + SHARED_THEME
    + gg.theme(legend_position="bottom", figure_size=(3, 2))
)
fig.save("min_hamming_dist_ecdf.png", dpi=500, bbox_inches="tight")
display(fig)

fig2 = (
    gg.ggplot(min_hamming_dist, gg.aes(x="min_hamming"))
    + gg.geom_histogram(bins=100)
    + gg.theme_minimal()
    + gg.labs(x="Minimum Hamming distance", y="Cumulative Density")
    + SHARED_THEME
    + gg.theme(legend_position="bottom", figure_size=(3, 2))
)
fig2.save("min_hamming_dist_hist.png", dpi=500, bbox_inches="tight")
display(fig2)

In [ ]:
g = plot_similarity_matrix(
    flux_ham_dist_df,
    growth_df,
    row_color_column="growth_ratio",
    similarity_label="flux cosine similarity",
)
# g.savefig("./flux_clustermap.png", dpi=500, bbox_inches="tight")
plt.show()

In [ ]:
tsne = TSNE_sklearn(n_components=2, metric="precomputed", random_state=42, init="random")
embedding = tsne.fit_transform(flux_ham_dist)

plot_df = pd.DataFrame(
    {
        "TSNE1": embedding[:, 0],
        "TSNE2": embedding[:, 1],
        "growth_ratio": growth_df.loc[flux_ham_dist_df.index, "growth_ratio"],
        "gene_name": flux_ham_dist_df.index,
    }
)

fig = px.scatter(
    plot_df,
    x="TSNE1",
    y="TSNE2",
    color="growth_ratio",
    hover_name="gene_name",
    template="plotly_white",
    color_continuous_scale="RdBu",
    width=600,
    height=400,
)
fig.update_traces(marker=dict(size=3))
fig.show()

In [ ]:
gene_subset = ["acpP", "msbA"]
flux_ham_dist_df.loc[gene_subset, gene_subset]

In [ ]:
gene_subset = ["lpxH", "waaA", "lptC", "kdsA", "lpxA", "lpxD"]
flux_ham_dist_df.loc[gene_subset, gene_subset]

In [ ]:
flux_ham_dist_df

In [ ]:
gene_subset = ["pdxK", "thiD", "solA", "ugpE", "nudC", "rhtB", "ugpC", "rhtA", "mhpE", "ligA"]
display(growth_df.loc[gene_subset, ["growth_ratio", "T4"]])

The Hamming distance 0 result reframes everything. Here's what it implies and what to watch for:

**What you'll almost certainly find**

- `growth_ratio ≈ 1.0` for ligA — the model predicts it as non-essential
- ligA will appear in very few reactions, likely just one: the NAD⁺-consuming nick-sealing reaction (`DNLJ` or similar). That reaction is a metabolic dead-end in the model — removing it doesn't affect the flux through any pathway that feeds biomass

**The core issue**

iJO1366 is a *metabolic* model. It has no representation of DNA replication integrity. ligA's essentiality is *topological* — without it, replication forks can't be completed — but this constraint is simply absent from the MILP. From the model's perspective, ligA is just a minor NAD⁺ sink, and knocking it out leaves growth completely unperturbed.

**What this means for your framework**

This is a high-value **false negative** from the model: predicted non-essential, experimentally essential. The Hamming distance 0 to genes like `ugpE` or `mhpE` is the model being *consistently wrong* — it has no mechanism to distinguish them.

Two things worth noting specifically:

1. **This class of genes** (DNA replication, cell division, transcription machinery) will systematically cluster with metabolically inert genes in MOMA space. It's a predictable blind spot you may want to handle explicitly — either by flagging genes whose essentiality *cannot* be explained by the metabolic model, or by using a separate gene essentiality annotation to partition your surprise detection.

2. **The surprise signal here is inverted**: rather than a model predicting a strong flux perturbation that isn't observed experimentally, ligA is a case where the model predicts *nothing* but biology says *lethal*. Your framework needs to handle surprises in both directions.

In [ ]:
growth_df.loc["ligA"]

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(3, 2))
triu_indices = np.triu_indices(flux_df_bin.shape[1], k=1)
plt.hist(flux_ham_dist[triu_indices], bins=100)
plt.xlabel("Hamming distance")
plt.ylabel("Count")
plt.title("Hamming distance between flux reactions (MOMA)")
plt.show()

### clustering ideas

In [ ]:
essential_genes = growth_df.loc[growth_df["growth_ratio"] < 0.5].index.tolist()
flux_ham_dist_df_essential = flux_ham_dist_df.loc[essential_genes, essential_genes]

#### Pairwise comparisons

In [ ]:
flux_ham_dist_df_essential_long = to_long_no_diagonal(flux_ham_dist_df_essential)
transcript_dist_selected_long = to_long_no_diagonal(transcript_pairwise)

joint_dists = flux_ham_dist_df_essential_long.merge(
    transcript_dist_selected_long,
    on=["gene1", "gene2", "gene_pair"],
    suffixes=["_flux", "_transcript"],
)

In [ ]:
(
    gg.ggplot(joint_dists, gg.aes(x="distance_flux", y="distance_transcript"))
    + gg.geom_point(alpha=0.1)
    + gg.theme_minimal()
    + gg.labs(x="Flux distance", y="Transcript distance")
    + SHARED_THEME
    + gg.theme(figure_size=(3, 2))
)

In [ ]:
joint_select = joint_dists.query("distance_flux < 10.0")
stats.spearmanr(joint_select["distance_flux"], joint_select["distance_transcript"])

In [ ]:
(
    gg.ggplot(
        joint_dists.query("distance_flux < 10.0"),
        gg.aes(x="distance_flux", y="distance_transcript"),
    )
    + gg.geom_point(alpha=0.1)
    + gg.theme_minimal()
    + gg.labs(x="Flux distance", y="Transcript distance")
    + SHARED_THEME
    + gg.theme(figure_size=(3, 2))
)

In [ ]:
fig = px.scatter(
    joint_dists.query("distance_flux < 10.0"),
    x="distance_flux",
    y="distance_transcript",
    hover_name="gene_pair",
    template="plotly_white",
    width=600,
    height=400,
)
fig.update_traces(marker=dict(size=3))
fig.show()

In [ ]:
joint_dists.query("distance_flux < 10.0").sort_values("distance_transcript", ascending=False).head(
    30
)

In [ ]:
flux_df_bin

In [ ]:
gene_subset = ["pdxK", "thiD", "solA", "ugpE", "nudC", "rhtB", "ugpC", "rhtA", "mhpE", "ligA"]
display(growth_df.loc[gene_subset, ["growth_ratio", "T4"]])

In [ ]:
flux_a = flux_df_bin.loc["coaA"]
flux_b = flux_df_bin.loc["folC"]
disagrees = flux_a != flux_b

print(flux_df_bin.columns[disagrees])

In [ ]:
gene1 = "hemA"
gene2 = "hemE"

obs_subset = adata_case.obs.loc[adata_case.obs["gene"].isin([gene1, gene2])]
obs_subset["gene"] = obs_subset["gene"].astype(str)
(
    gg.ggplot(adata_case.obs, gg.aes(x="transcript_case_UMAP1", y="transcript_case_UMAP2"))
    + gg.geom_point(alpha=0.1)
    + gg.geom_point(
        obs_subset,
        gg.aes(x="transcript_case_UMAP1", y="transcript_case_UMAP2", color="gene"),
    )
    + gg.theme_minimal()
    + gg.labs(x="UMAP1", y="UMAP2")
    + gg.theme(figure_size=(3, 2))
)

In [ ]:
joint_dists.loc[lambda x: x["gene1"].isin(["lpxA", "lpxD", "lpxK", "lpxB"])].loc[
    lambda x: x["gene2"].isin(["lpxA", "lpxD", "lpxK", "lpxB"])
].plot.scatter(x="distance_flux", y="distance_transcript")
plt.show()

In [ ]:
fig = px.scatter(
    joint_dists,
    x="distance_flux",
    y="distance_transcript",
    hover_name="gene_pair",
    template="plotly_white",
    width=600,
    height=400,
)
fig.update_traces(marker=dict(size=3))
fig.show()

#### Hierarchical clustering on flux Hamming distance

In [ ]:
import pandas as pd
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, fcluster

condensed_dist = squareform(flux_ham_dist_df_essential.values)
Z = linkage(condensed_dist, method="complete")
distance_threshold = 20.0
clusters_dist = fcluster(Z, t=distance_threshold, criterion="distance")
cluster_assignments = pd.Series(clusters_dist, index=flux_ham_dist_df_essential.index)

In [ ]:
unique_clusters = sorted(cluster_assignments.unique())
palette = sns.color_palette("tab10", n_colors=len(unique_clusters))
cluster_color_map = dict(zip(unique_clusters, palette))

row_colors = cluster_assignments.map(cluster_color_map)

g = sns.clustermap(
    flux_ham_dist_df_essential,
    row_linkage=Z,
    col_linkage=Z,
    row_colors=row_colors,
    col_colors=row_colors,
    cmap="rocket_r",
    xticklabels=False,
    yticklabels=False,
    cbar_pos=(0.02, 0.8, 0.05, 0.18),
    cbar_kws={"label": "Hamming Distance"},
)

legend_patches = [
    mpatches.Patch(color=color, label=f"Cluster {cluster_id}")
    for cluster_id, color in cluster_color_map.items()
]

# Place the legend slightly outside the heatmap
g.ax_heatmap.legend(
    handles=legend_patches,
    title="Clusters",
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=False,
)

plt.show()

In [ ]:
for cluster_id in cluster_assignments.unique():
    gene_list = cluster_assignments[cluster_assignments == cluster_id].index.tolist()

    gene_inter = list(
        set(gene_list) & set(flux_ham_dist_df_essential.index) & set(transcript_pairwise.index)
    )
    if len(gene_inter) <= 1:
        continue
    print(",".join(gene_inter))
    print()

    flux_dist_selected = flux_ham_dist_df_essential.loc[gene_inter, gene_inter]
    transcript_dist_selected = transcript_pairwise.loc[gene_inter, gene_inter]

    # g_flux = sns.clustermap(
    #     flux_dist_selected,
    #     cmap="rocket_r",
    #     xticklabels=False,
    #     yticklabels=True,
    #     vmax=np.quantile(flux_dist_selected.values, 0.9),
    # )
    # plt.show()

    # sns.clustermap(
    #     transcript_dist_selected,
    #     cmap="rocket_r",
    #     xticklabels=False,
    #     yticklabels=True,
    #     row_linkage=g_flux.dendrogram_row.linkage,
    #     col_linkage=g_flux.dendrogram_col.linkage,
    #     vmax=np.quantile(transcript_dist_selected.values, 0.9),
    # )
    # plt.show()

    trans_dist_selected_long = to_long_no_diagonal(transcript_dist_selected)
    flux_dist_selected_long = to_long_no_diagonal(flux_dist_selected)
    joint_dists = trans_dist_selected_long.merge(
        flux_dist_selected_long,
        on=["gene1", "gene2"],
        suffixes=["_transcript", "_flux"],
    )

    fig = (
        gg.ggplot(joint_dists, gg.aes(x="distance_transcript", y="distance_flux"))
        + gg.geom_point(alpha=0.1)
        + gg.theme_minimal()
        + gg.labs(x="Transcript distance", y="Flux distance")
        + SHARED_THEME
        + gg.theme(figure_size=(3, 2))
    )
    # fig.save("joint_dists.png", dpi=500, bbox_inches="tight")
    display(fig)